In [1]:
import csv
import os
import shutil
import re

from lab_utils import INTEGRITY_TABLE, BACKUP_FOLDER, FOLDER
from lab_utils import append_integrity_file, calculate_hash, get_current_timestamp

Part 1 - Create a folder of files to work with

In [2]:
# Reset project to default state
if os.path.exists(INTEGRITY_TABLE):
    os.remove(INTEGRITY_TABLE)

shutil.rmtree(FOLDER, ignore_errors=True)
os.makedirs(FOLDER)

if os.path.exists(BACKUP_FOLDER):
    shutil.copytree(BACKUP_FOLDER, FOLDER, dirs_exist_ok=True)

Part 2 - Create Integrity CSV table

In [3]:
# make an integrity table for all files in the working folder
for root, dirs, files in os.walk(FOLDER):
    for file in files:
        file_path = os.path.join(root, file)
        hash_value = calculate_hash(file_path)
        timestamp = get_current_timestamp()
        append_integrity_file(file_path, hash_value, timestamp)

Part 3 - Validate integrity table

In [4]:
# change a file to simulate tampering. here i will change simple.txt contents
bad_file_path = os.path.join(FOLDER, "simple.txt")
with open(bad_file_path, "w", encoding="utf-8") as f:
    f.write("This file is no longer containing the original text. :c")

# delete a file. house.png will get blown up
bad_file_path = os.path.join(FOLDER, "house.png")
if os.path.exists(bad_file_path):
    os.remove(bad_file_path)

# read integrity table and check if all files are intact
with open(INTEGRITY_TABLE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        file_path = row["file_path"]
        expected_hash = row["sha256_digest"]
        if os.path.exists(file_path):
            actual_hash = calculate_hash(file_path)
            if actual_hash != expected_hash:
                print(f"File {file_path} is modified")
            else:
                print(f"File {file_path} is intact")
        else:
            print(f"File {file_path} is missing")

File files\bridge.png is intact
File files\dark.png is intact
File files\evil.py is intact
File files\house.png is missing
File files\road.png is intact
File files\simple.txt is modified
File files\others\desert.png is intact


Part 4 - Detect Malware using suspicious patterns

In [5]:
SIGNATURES = [
    r"eval\(", 
    r"base64\.b64decode",
    r"socket\.connect",
    r"exec\(",
    r"import os"
]

EVIL_FILE = os.path.join(FOLDER, "evil.py")

with open(EVIL_FILE, "r", encoding="utf-8") as f:
    content = f.read()
    for signature in SIGNATURES:
        if re.search(signature, content):
            print(f"Signature '{signature}' found in {EVIL_FILE}")

Signature 'eval\(' found in files\evil.py
Signature 'base64\.b64decode' found in files\evil.py
Signature 'socket\.connect' found in files\evil.py
Signature 'exec\(' found in files\evil.py
Signature 'import os' found in files\evil.py
